# Convertible-Bonds — a quantitative teardown 🔬
### Quadratic convexity regression · HAC inference · block bootstrap · matched-blend race · capacity

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Equity_upside_bond_downside%3F: Busted](https://img.shields.io/badge/Equity_upside_bond_downside%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We test the convertibles convexity claim as a **sign-and-significance** question about the quadratic coefficient of the payoff, and as a **replication** question against a beta-matched stock/bond blend.

> ⚠️ **Not investment advice.** Real tape: CWB / SPY / AGG / SHY total return (`yfinance auto_adjust=True`), cache-first; offline fallback is a planted-convex synthetic control (machinery proof only). Methods + sources in [`docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

from convertible_bonds import data, strategy as st


def load_tape():
    """Real CWB/SPY/AGG/SHY frame if reachable, else the deterministic synthetic tape."""
    try:
        frame = data.load_real(end="2026-05-31")
        if len(frame) > 1000 and {"CWB", "SPY", "AGG", "SHY"}.issubset(frame.columns):
            return frame, "real"
    except Exception as e:
        print("real load failed -> synthetic:", type(e).__name__)
    frame, _ = data.synthetic_convex(n_days=4000, gamma=0.6, seed=339)
    return frame, "synthetic"


FRAME, TAPE = load_tape()
# Column map: real tape uses tickers; synthetic uses IDX/BND/CVT.
if TAPE == "real":
    CVT, IDX, BND, CASH = "CWB", "SPY", "AGG", "SHY"
    BANNER = "REAL tape — CWB / SPY / AGG / SHY, total return"
else:
    CVT, IDX, BND, CASH = "CVT", "IDX", "BND", None
    BANNER = "SYNTHETIC tape (offline fallback — a PLANTED-convex control, NOT real CWB)"
RETS = st.to_returns(FRAME)
RF = RETS[CASH] if CASH else None
print("tape in this run:", BANNER)
print("rows:", len(FRAME), " range:", FRAME.index[0].date(), "->", FRAME.index[-1].date())

from convertible_bonds import data, strategy as st


tape in this run: REAL tape — CWB / SPY / AGG / SHY, total return
rows: 4307  range: 2009-04-16 -> 2026-05-29


## Verdict, up front

*Real CWB/SPY/AGG/SHY tape, 2009-04-16 → 2026-05-29 (4,307 days, panel fp `0ab3e0356db0`), from [docs/results.md](../docs/results.md).*

| Axis | Stamp | The decisive number |
|---|---|---|
| **Signal** — is the payoff convex? | **None** | Quadratic gamma = **-1.897** (HAC *t* = **-2.96**) — *negative*, the wrong sign; bootstrap CI **[-3.484, -0.873]** wholly **below** 0. |
| **Tradability** — better than a plain blend? | **Mirage** | A 63%/37% SPY/AGG blend matched the beta and **edged** CWB on excess Sharpe (0.879 vs 0.817); the diff is insignificant (*t* = 0.71). |
| **Equity upside, bond downside?** | **Busted** | Up-capture 0.664 ≈ down-capture 0.645 (asym +0.019); in Mar-2020 CWB fell -13.1% vs SPY -12.5% — no floor. |

> 💡 **In plain words:** the curve that *defines* a convertible — bend up, catch the rally, cushion the fall — is simply not in the CWB tape. What's left is a stock/bond blend you can buy for less.

## 1 · The claim, steelmanned

Three testable hypotheses behind *'equity upside, bond downside'*:

- **H₁ (convexity):** the convertible's return is a *convex* function of the equity index — the coefficient on `up² = max(r_idx, 0)²` is **> 0** with a robust *t* ≥ 2.
- **H₂ (asymmetry):** up-capture is materially **above** down-capture.
- **H₃ (irreplaceable):** a beta-matched linear stock/bond blend **cannot** match the convertible's risk-adjusted return.

Convexity is real here only if **all three** hold. We reject if gamma isn't positive-significant, capture is roughly symmetric, and the blend keeps pace.

## 2 · So what? — what rides on each answer

Convexity is the *only* thing a convertible offers that a stock and a bond, held separately, do not — a non-linear, asymmetric payoff. If H₁–H₃ hold, the ETF is a cheap way to rent gamma. If they fail, the category is a **linear stock/bond exposure with an option premium and a fund fee bolted on** — negative expected value versus the do-it-yourself blend, dressed as a free lunch.

## 3 · How we'd know — the protocol

1. **Decompose** the payoff: OLS of `r_cvt` on `[1, r_idx, up²]`. Gamma is the convexity; we recover it by Frisch-Waugh-Lovell and cross-check.
2. **Robust inference:** a Newey-West (HAC) *t* on gamma (returns are autocorrelated and heteroskedastic), plus a **circular block bootstrap** CI on gamma that preserves volatility clustering.
3. **Asymmetry:** up-/down-capture vs the index.
4. **Alpha vs beta / replication:** a beta-matched annual-rebalance stock/bond blend, raced on **excess-of-cash** Sharpe (SHY proxy) with a HAC *t* and block bootstrap on the Sharpe *difference*.
5. **Capacity / cost:** the replica is two of the cheapest ETFs alive; CWB charges ~0.40%/yr — the break-even the convexity would have to beat.
6. **Verdict** — the stamps with their numbers.

> 💡 **In plain words:** we don't just check *if* the convertible went up with stocks — we check whether it went up *faster than proportionally*, which is the actual mathematical content of 'convexity'.

## 4 · The teardown

### 4.1 The convexity regression (H₁)

`r_cvt = α + β·r_idx + γ·up²`. γ is the bend; we want γ > 0, robust *t* ≥ 2.

In [2]:
reg = st.convexity_regression(RETS, CVT, IDX)
print(BANNER)
print(f'beta (linear)   : {reg["beta"]:.4f}')
print(f'gamma (convex.) : {reg["gamma"]:.4f}   FWL check {reg["gamma_fwl"]:.4f}')
print(f'HAC t(gamma)    : {reg["t_gamma"]:.2f}')
print(f'R^2             : {reg["r2"]:.3f}')
verdict = 'CONVEX (smile)' if reg['gamma']>0 and reg['t_gamma']>2 else \
          ('CONCAVE (frown)' if reg['gamma']<0 and reg['t_gamma']<-2 else 'flat / noisy')
print('payoff shape    :', verdict)

REAL tape — CWB / SPY / AGG / SHY, total return
beta (linear)   : 0.6584
gamma (convex.) : -1.8975   FWL check -1.8975
HAC t(gamma)    : -2.96
R^2             : 0.692
payoff shape    : CONCAVE (frown)


> 💡 **In plain words:** on the real CWB tape γ comes out **negative and significant** — the payoff bends *down*, not up. That is the single cleanest rejection of the convexity thesis: not 'we found nothing', but 'we found the **opposite**'. (The synthetic fallback plants γ > 0 so the harness can prove it *detects* convexity when it's really there — a machinery check, never evidence.)

### 4.2 Bootstrap CI on gamma

Does the bend's sign survive resampling?

In [3]:
bg = st.bootstrap_gamma(RETS, CVT, IDX, block=21, n_boot=1000, seed=339)
print(f'gamma point : {bg["point"]:+.3f}')
print(f'95% CI      : [{bg["ci95"][0]:+.3f}, {bg["ci95"][1]:+.3f}]')
print(f'P(gamma>0)  : {bg["frac_pos"]:.2%}')

gamma point : -1.897
95% CI      : [-3.540, -0.871]
P(gamma>0)  : 0.00%


### 4.3 Capture asymmetry (H₂)

Convexity should show as up-capture well above down-capture.

In [4]:
cap = st.capture_ratios(RETS, CVT, IDX)
print(f'up-capture   : {cap["up_capture"]:.3f}  (n={cap["n_up"]})')
print(f'down-capture : {cap["down_capture"]:.3f}  (n={cap["n_dn"]})')
print(f'asymmetry    : {cap["asymmetry"]:+.3f}')

up-capture   : 0.664  (n=2389)
down-capture : 0.645  (n=1905)
asymmetry    : +0.019


### 4.4 The replication race (H₃)

Beta-match a stock/bond blend to the convertible, race on **excess-of-cash** Sharpe, then bootstrap the Sharpe *difference* (CWB − blend).

In [5]:
beta = st.beta_to(RETS, CVT, IDX); w = float(np.clip(beta, 0, 1))
blend = st.matched_blend(RETS, IDX, BND, w_stock=w, rebalance='annual', cost_bps=2.0)
rows = []
for nm, s in [(CVT, RETS[CVT]), ('matched blend', blend), (IDX, RETS[IDX]), (BND, RETS[BND])]:
    d = st.stats(s, rf=RF)
    rows.append((nm, d['cagr'], d['vol'], d['sharpe'], d['max_dd']))
tbl = pd.DataFrame(rows, columns=['arm','CAGR','vol','Sharpe_xs','maxDD']).set_index('arm')
print(f'matched weight w_stock = {w:.2f} (= CWB beta to {IDX})')
tbl

matched weight w_stock = 0.63 (= CWB beta to SPY)


,CAGR,vol,Sharpe_xs,maxDD
arm,,,,
CWB,0.120,0.131,0.817,-0.321
matched blend,0.110,0.109,0.879,-0.220
SPY,0.156,0.172,0.838,-0.337
AGG,0.028,0.047,0.378,-0.184


In [6]:
bs = st.bootstrap_sharpe_diff(RETS[CVT], blend, rf=RF, block=21, n_boot=1000, seed=339)
dr = (RETS[CVT] - blend).dropna()
print(f'Sharpe diff (CWB - blend): {bs["point"]:+.3f}')
print(f'95% CI                  : [{bs["ci95"][0]:+.3f}, {bs["ci95"][1]:+.3f}]')
print(f'CWB wins the race in    : {bs["frac_a_wins"]:.0%} of resamples')
print(f'HAC t on daily (CWB-blend) return diff: {st.hac_tstat(dr.to_numpy()):.2f}')

Sharpe diff (CWB - blend): -0.062
95% CI                  : [-0.297, +0.183]
CWB wins the race in    : 30% of resamples
HAC t on daily (CWB-blend) return diff: 0.71


> 💡 **In plain words:** the matched blend's Sharpe sits *above* CWB's and the difference is statistically a coin flip — so CWB is, at best, indistinguishable from the cheap DIY mix, and on the point estimate a touch worse. H₃ fails.

## 5 · The verdict

| Hypothesis | Test | Real CWB tape | Holds? |
|---|---|---|---|
| H₁ convexity | γ > 0, HAC *t* ≥ 2 | γ = **-1.897**, *t* = **-2.96** (CI [-3.484, -0.873]) | ❌ (*wrong sign*) |
| H₂ asymmetry | up-cap ≫ down-cap | 0.664 vs 0.645 (asym +0.019) | ❌ |
| H₃ irreplaceable | blend can't match | blend Sharpe 0.879 ≥ CWB 0.817, diff *t* = 0.71 | ❌ |

All three fail. **Signal None · Tradability Mirage · 'Equity upside, bond downside' Busted.** Robust across halves: γ stays negative in 2009–2015 (-1.847, *t* -1.56) and 2016–2026 (-2.322, *t* -3.4).

## 6 · Could you trade it?

There's nothing to *trade* — the question is whether to *own* CWB over the DIY replica. CWB charges ~0.40%/yr; the replica is a broad-stock ETF + a broad-bond ETF at single-digit-bp fees. For CWB to be worth the wrapper it would need to deliver convexity worth more than ~0.40%/yr — but the measured convexity is *negative*, so the fee buys you a *worse* shape than the blend. Capacity isn't the constraint (CWB is liquid); **economics** is.

## 7 · Going further

- **Single-name decomposition.** An ETF averages over balanced / busted / equity-like converts; fit γ on individual issues bucketed by moneyness — convexity may live where the basket washes it out.
- **Conditional γ.** Roll the quadratic regression; does the bend turn positive in calm bull regimes and negative in crashes (i.e. *negative* gamma exactly when you wanted the floor)?
- **Credit leg.** Decompose CWB into equity + rates + credit factors; how much of the 'bond floor' is just HY credit beta (which sells off *with* equities)?
- **Other wrappers:** ICVT, FCVT — same harness, swap the ticker.

Fork it, change the ticker in `data.load_real`, re-run.